# 第27章　ドメイン適応 ― 施設・装置の違いを乗り越える

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 具体手法を、3段で

In [ ]:
import torch

# DANNの要: 勾配反転で「施設を当てられない特徴」を作る
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd): ctx.lambd = lambd; return x
    @staticmethod
    def backward(ctx, g): return -ctx.lambd * g, None   # 勾配を反転

feat = encoder(x)
y_pred = classifier(feat)                    # 病変分類は正しく学ぶ
d_pred = domain_head(GradReverse.apply(feat, lambd))  # 施設は当てさせない
loss = task_loss(y_pred, y) + domain_loss(d_pred, domain_label)

## 推論のその場で適応する ― テスト時適応（TTA/TENT）

In [ ]:
configure_bn_only(model)                 # γ,β だけ requires_grad=True
for batch in test_stream:                # 推論しながら、その場で適応
    logits = model(batch)
    loss = softmax_entropy(logits).mean() # 確信度を高める方向へ
    loss.backward(); optimizer.step(); optimizer.zero_grad()
    preds = logits.argmax(1)   # 予測は更新前の値（適応は次バッチから）